# 🏡 House Price Prediction - Comprehensive Machine Learning Workflow

This notebook demonstrates an end-to-end Machine Learning pipeline to predict residential house prices:
1. **Data Loading & Inspection**: Examining property features (Area, Bedrooms, Bathrooms, Location, Year Built, etc.).
2. **Data Preprocessing & Cleaning**: Handling missing values, removing duplicate records, and outlier analysis.
3. **Exploratory Data Analysis (EDA)**: Visualizing distributions, correlations, and relationships.
4. **Pipeline Construction**: Preventing data leakage using `ColumnTransformer` and `Pipeline`.
5. **Model Training & Comparison**: Linear Regression, Decision Tree, Random Forest, and Gradient Boosting.
6. **Evaluation & Cross-Validation**: Computing MAE, MSE, RMSE, and R² scores.
7. **Model Serialization**: Saving the best performing pipeline using `joblib`.
8. **Inference**: Testing predictions on new unseen property listings.

In [ ]:
import os
import sys
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set modern visual aesthetics
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.sans-serif"] = "DejaVu Sans"

# Adjust system path to import src modules
sys.path.append(os.path.abspath(os.path.join("..", "src")))
from data_preprocessing import load_dataset, remove_duplicates, check_missing_values, create_preprocessor
from train_model import train_and_compare_models, save_best_model
from predict import HousePricePredictor

print("All modules imported successfully!")

## 1. Data Loading & Inspection

In [ ]:
data_path = os.path.join("..", "data", "house_data.csv")
df = pd.read_csv(data_path)
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

In [ ]:
# Descriptive statistics of numerical features
df.describe().T

## 2. Data Cleaning & Sanity Checks
- Check and remove duplicates
- Check missing values

In [ ]:
# Check duplicates
dup_count = df.duplicated().sum()
print(f"Duplicate records found: {dup_count}")
df_clean = df.drop_duplicates().copy()
print(f"Shape after dropping duplicates: {df_clean.shape}")

# Missing values analysis
missing = df_clean.isnull().sum()
print("\nMissing values per column:")
print(missing[missing > 0])

## 3. Exploratory Data Analysis (EDA)
Let's create the key visualizations to understand market trends and relationships.

In [ ]:
# 1. Price Distribution
fig, ax = plt.subplots(figsize=(8, 5))
sns.histplot(df_clean["Price"], kde=True, color="#3b82f6", bins=35, ax=ax, edgecolor="white", alpha=0.6)
mean_val = df_clean["Price"].mean()
median_val = df_clean["Price"].median()
ax.axvline(mean_val, color="#ef4444", linestyle="--", label=f"Mean: ${mean_val:,.0f}")
ax.axvline(median_val, color="#10b981", linestyle="-.", label=f"Median: ${median_val:,.0f}")
ax.set_title("House Price Distribution", fontsize=14, fontweight="bold")
ax.xaxis.set_major_formatter("${x:,.0f}")
ax.legend()
plt.show()

In [ ]:
# 2. Area vs. Price
fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(data=df_clean, x="Area", y="Price", hue="PropertyType", palette="viridis", alpha=0.7, ax=ax)
sns.regplot(data=df_clean, x="Area", y="Price", scatter=False, color="red", ax=ax, line_kws={"linewidth": 2})
ax.set_title("Living Area vs. House Price", fontsize=14, fontweight="bold")
ax.yaxis.set_major_formatter("${x:,.0f}")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.show()

In [ ]:
# 3. Bedrooms vs. Price
fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=df_clean, x="Bedrooms", y="Price", palette="Blues_r", hue="Bedrooms", legend=False, ax=ax)
ax.set_title("Bedrooms vs. House Price", fontsize=14, fontweight="bold")
ax.yaxis.set_major_formatter("${x:,.0f}")
plt.show()

In [ ]:
# 4. Bathrooms vs. Price
fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=df_clean, x="Bathrooms", y="Price", palette="Purples_r", hue="Bathrooms", legend=False, ax=ax)
ax.set_title("Bathrooms vs. House Price", fontsize=14, fontweight="bold")
ax.yaxis.set_major_formatter("${x:,.0f}")
plt.show()

In [ ]:
# 5. Correlation Heatmap
fig, ax = plt.subplots(figsize=(8, 6))
numeric_corr = df_clean.select_dtypes(include=[np.number]).corr()
mask = np.triu(np.ones_like(numeric_corr, dtype=bool))
sns.heatmap(numeric_corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Numeric Features Correlation Matrix", fontsize=14, fontweight="bold")
plt.show()

In [ ]:
# 6. Location vs. Average Price
fig, ax = plt.subplots(figsize=(8, 5))
order = df_clean.groupby("Location")["Price"].median().sort_values(ascending=False).index
sns.barplot(data=df_clean, x="Location", y="Price", order=order, palette="mako", hue="Location", legend=False, ax=ax)
ax.set_title("Average House Price by Location", fontsize=14, fontweight="bold")
ax.yaxis.set_major_formatter("${x:,.0f}")
plt.xticks(rotation=15)
plt.show()

## 4. Model Training, 5-Fold Cross-Validation & Evaluation
We train and compare four regression models:
- **Linear Regression**
- **Decision Tree Regressor**
- **Random Forest Regressor**
- **Gradient Boosting Regressor**

Using an isolated `ColumnTransformer` inside a `Pipeline` to strictly avoid data leakage.

In [ ]:
comparison_df, best_model_name, best_pipeline, all_pipelines = train_and_compare_models(data_path)
comparison_df

## 5. Serializing Best Model Pipeline with Joblib

In [ ]:
models_folder = os.path.join("..", "models")
saved_path = save_best_model(best_pipeline, best_model_name, comparison_df, models_folder)
print(f"Best model stored at: {saved_path}")

## 6. End-to-End Prediction Inference on New Property Listings

In [ ]:
predictor = HousePricePredictor(saved_path)

new_house = {
    "Area": 3200,
    "Bedrooms": 4,
    "Bathrooms": 3,
    "Floors": 2,
    "Parking": 2,
    "Location": "Waterfront",
    "YearBuilt": 2020,
    "PropertyType": "Single Family"
}

prediction_result = predictor.predict(new_house)

print("=== Valuation Summary ===")
print(f"Estimated House Price: {prediction_result['formatted_price']}")
print(f"Estimated Price Range: {prediction_result['formatted_range']}")
print(f"Price per Square Foot: ${prediction_result['price_per_sqft']:,.2f}")